# Cox proportional hazards with left truncation

Rows are `(start, stop]` intervals. The partial likelihood uses `start < t <= stop`, so left-truncated rows and stacked time-varying covariates share the same layout.

The GBNet fit below supplies only a PyTorch loss. It does not manually provide gradients or Hessians.

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from lifelines.utils import concordance_index
from matplotlib import pyplot as plt

from gbnet.xgbmodule import XGBModule


class CoxPHGBNet:
    def __init__(self, nrounds=40, params=None, min_hess=1e-2):
        self.nrounds = nrounds
        self.params = {} if params is None else params
        self.min_hess = min_hess
        self.losses_ = []

    def _prepare_risk_sets(self, y):
        starts = torch.tensor(y["start"].to_numpy(), dtype=torch.float32)
        stops = torch.tensor(y["stop"].to_numpy(), dtype=torch.float32)
        events = y["event"].to_numpy().astype(bool)

        failure_times, event_group, event_counts = np.unique(
            y.loc[events, "stop"].to_numpy(),
            return_inverse=True,
            return_counts=True,
        )

        failure_times = torch.tensor(failure_times, dtype=torch.float32)
        self.risk_mask_ = (starts[None, :] < failure_times[:, None]) & (
            stops[None, :] >= failure_times[:, None]
        )
        self.failure_times_ = failure_times
        self.event_index_ = torch.tensor(np.flatnonzero(events), dtype=torch.long)
        self.event_group_ = torch.tensor(event_group, dtype=torch.long)
        self.event_counts_ = torch.tensor(event_counts, dtype=torch.float32)

    def _negative_partial_log_likelihood(self, eta):
        eta = eta.flatten()
        event_eta_by_time = eta.new_zeros(len(self.failure_times_)).scatter_add(
            0, self.event_group_, eta[self.event_index_]
        )
        log_risk_sum = torch.logsumexp(
            eta.unsqueeze(0).masked_fill(~self.risk_mask_, -torch.inf), dim=1
        )
        log_lik = event_eta_by_time.sum() - (
            self.event_counts_ * log_risk_sum
        ).sum()
        return -log_lik / len(eta)

    def fit(self, X, y):
        self.feature_cols_ = list(X.columns)
        self._prepare_risk_sets(y)

        dtrain = xgb.DMatrix(X[self.feature_cols_])
        self.model_ = XGBModule(
            batch_size=X.shape[0],
            input_dim=X.shape[1],
            output_dim=1,
            params=self.params,
            min_hess=self.min_hess,
        )

        self.losses_ = []
        for _ in range(self.nrounds):
            self.model_.train()
            self.model_.zero_grad()

            eta = self.model_(dtrain).flatten()
            loss = self._negative_partial_log_likelihood(eta)
            loss.backward(create_graph=True)
            self.losses_.append(loss.item())

            self.model_.gb_step()
            self.model_.FX.grad = None

        self.model_.eval()
        return self

    def predict_log_hazard(self, X):
        self.model_.eval()
        return self.model_(xgb.DMatrix(X[self.feature_cols_]), return_tensor=False).reshape(-1)

    def predict_relative_risk(self, X):
        return np.exp(self.predict_log_hazard(X))

In [ ]:
rng = np.random.default_rng(7)
n = 1500

age = rng.normal(size=n)
treatment = rng.binomial(1, 0.5, size=n)
marker0 = rng.normal(size=n)
change = rng.uniform(0.5, 3.0, size=n)
marker1 = marker0 + 0.8 * treatment + rng.normal(scale=0.5, size=n)

beta = np.array([0.7, -0.5, 0.6])
eta0 = beta[0] * age + beta[1] * treatment + beta[2] * marker0
eta1 = beta[0] * age + beta[1] * treatment + beta[2] * marker1

rate0 = 0.08 * np.exp(eta0)
rate1 = 0.08 * np.exp(eta1)
wait = rng.exponential(size=n)
event_time = np.where(
    wait <= rate0 * change,
    wait / rate0,
    change + (wait - rate0 * change) / rate1,
)
censor_time = rng.uniform(2.0, 8.0, size=n)
observed_time = np.minimum(event_time, censor_time)
event = event_time <= censor_time
entry = rng.uniform(0.0, 1.5, size=n) * (rng.random(n) < 0.35)

subjects = pd.DataFrame(
    {
        "unit_id": np.arange(n),
        "entry": entry,
        "change": change,
        "stop": observed_time,
        "event": event,
        "age": age,
        "treatment": treatment,
        "marker0": marker0,
        "marker1": marker1,
        "eta0": eta0,
        "eta1": eta1,
    }
)
subjects = subjects[subjects["stop"] > subjects["entry"]].copy()

pre = subjects[
    subjects["entry"] < np.minimum(subjects["change"], subjects["stop"])
].copy()
pre["start"] = pre["entry"]
pre["end"] = np.minimum(pre["change"], pre["stop"])
pre["marker"] = pre["marker0"]
pre["row_event"] = pre["event"] & (pre["stop"] <= pre["change"])
pre["true_eta"] = pre["eta0"]

post = subjects[
    subjects["stop"] > np.maximum(subjects["entry"], subjects["change"])
].copy()
post["start"] = np.maximum(post["entry"], post["change"])
post["end"] = post["stop"]
post["marker"] = post["marker1"]
post["row_event"] = post["event"] & (post["stop"] > post["change"])
post["true_eta"] = post["eta1"]

intervals = (
    pd.concat([pre, post], ignore_index=True)
    .sort_values(["unit_id", "start"])
    .reset_index(drop=True)
)
X = intervals[["age", "treatment", "marker"]].copy()
y = intervals[["start", "end", "row_event"]].rename(
    columns={"end": "stop", "row_event": "event"}
).copy()

intervals.shape, int(y["event"].sum()), y["start"].gt(0).mean()

In [ ]:
default_dtrain = xgb.DMatrix(
    X.head(4), label=np.array([1.0, -2.0, 3.0, -4.0])
)
default_booster = xgb.train(
    {"objective": "survival:cox"}, default_dtrain, num_boost_round=0
)
default_config = json.loads(default_booster.save_config())

pd.Series(
    {
        **default_config["learner"]["gradient_booster"]["tree_train_param"],
        "tree_method": default_config["learner"]["gradient_booster"][
            "gbtree_train_param"
        ]["tree_method"],
        "updater": default_config["learner"]["gradient_booster"][
            "gbtree_train_param"
        ]["updater"],
        "base_score": default_config["learner"]["learner_model_param"][
            "base_score"
        ],
        "round0_margin": default_booster.predict(
            default_dtrain, output_margin=True
        )[0],
    }
)[
    [
        "eta",
        "max_depth",
        "min_child_weight",
        "lambda",
        "alpha",
        "subsample",
        "colsample_bytree",
        "max_delta_step",
        "tree_method",
        "updater",
        "base_score",
        "round0_margin",
    ]
]

In [ ]:
conservative_xgb_params = {
    "eta": 0.05,
    "max_depth": 2,
    "min_child_weight": 1,
    "max_delta_step": 0.1,
    "seed": 0,
    "nthread": 1,
}

model = CoxPHGBNet(nrounds=40, params=conservative_xgb_params, min_hess=1e-2)
model.fit(X, y)

intervals["eta_hat"] = model.predict_log_hazard(X)
last_interval = intervals.groupby("unit_id", as_index=False).tail(1)

pd.Series(
    {
        "first_loss": model.losses_[0],
        "last_loss": model.losses_[-1],
        "c_index": concordance_index(
            last_interval["stop"], -last_interval["eta_hat"], last_interval["event"]
        ),
        "front_loaded_alive_entries": int(model.risk_mask_.sum()),
    }
)

In [ ]:
# Native XGBoost Cox has no delayed-entry field, so compare on start = 0 data.
X_static = subjects[["age", "treatment", "marker0"]].rename(
    columns={"marker0": "marker"}
).copy()
y_static = pd.DataFrame(
    {
        "start": np.zeros(len(subjects)),
        "stop": subjects["stop"].to_numpy(),
        "event": subjects["event"].to_numpy(),
    }
)


def static_comparison(name, params, min_hess=1e-2):
    gbnet_cox = CoxPHGBNet(nrounds=40, params=params, min_hess=min_hess).fit(
        X_static, y_static
    )
    native_dtrain = xgb.DMatrix(
        X_static,
        label=np.where(y_static["event"], y_static["stop"], -y_static["stop"]),
    )
    native_cox = xgb.train(
        {**params, "objective": "survival:cox", "base_score": 1},
        native_dtrain,
        num_boost_round=40,
    )
    gbnet_margin = gbnet_cox.predict_log_hazard(X_static)
    native_margin = native_cox.predict(native_dtrain, output_margin=True)
    return {
        "setting": name,
        "gbnet_c_index": concordance_index(
            y_static["stop"], -gbnet_margin, y_static["event"]
        ),
        "native_c_index": concordance_index(
            y_static["stop"], -native_margin, y_static["event"]
        ),
        "margin_corr": np.corrcoef(gbnet_margin, native_margin)[0, 1],
        "gbnet_first_loss": gbnet_cox.losses_[0],
        "gbnet_last_loss": gbnet_cox.losses_[-1],
    }


pd.DataFrame(
    [
        static_comparison("xgboost defaults", {}),
        static_comparison("conservative shared params", conservative_xgb_params),
    ]
)

In [ ]:
fig, ax = plt.subplots()
ax.plot(model.losses_)
ax.set_xlabel("boosting round")
ax.set_ylabel("negative partial log likelihood")
fig